# Table: Overall Dataset Statistics

Generates the LaTeX table with per-snapshot statistics:
- **Complete dataset**: instances and TKGU operations
- **Subsampled test set**: instances and TKGU operations
- **KG statistics**: entities (from Wikipedia dictionaries), relation types and triples (from KG snapshot TSVs)

**Data sources:**
- Dataset stats: from `df_preds_gt_cie` (complete dataset + subsampled test set pkls)
- KG entities: line count of `dictionary_v3/YYYY-01-01-wiki_dictionary.jsonl`
- KG triples + rel types: from `kg_snapshot_YYYY-01-01.tsv`

In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.abspath('../../..'), 'src'))

from stats.evaluation.load_results import load_from_wiki_eval_result

In [2]:
# --- CONFIGURE THIS ---

# Subsampled test set (refactored pkl)
SUBSAMPLED_PKL = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260217_submitted_icml_fixed/wiki_eval_result.pkl'

# Complete dataset (old pipeline pkls)
# TODO: replace with refactored pkl once full-dataset pipeline is ported
COMPLETE_DATASET_DIR = '/path/to/storage/emerge/output/experiments/dataset_stats_pkls/20260124_all_dataset_no_models_v13_all/'

# KG snapshot TSVs (for triples and relation types)
KG_SNAPSHOTS_DIR = '/path/to/storage/wikidata-processing/output/experiments/s05_extract_wikidata_kg_snapshots/20250329/'

# Wikipedia entity dictionaries (for entity counts)
DICT_DIR = '/path/to/storage/wikipedia-processing/output/experiments/s08_extract_relik_index/20250910_slurm/dictionary_v3/'

SNAPSHOT_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

## Compute dataset stats (instances + operations per snapshot)

In [3]:
# Load subsampled test set
(_, _, df_gt_sub, _, df_inst_sub, _) = load_from_wiki_eval_result(SUBSAMPLED_PKL)
df_gt_sub = df_gt_sub[df_gt_sub['triple_type'] == 'in-dataset']
if 'snapshot_year' not in df_gt_sub.columns:
    df_gt_sub = df_gt_sub.merge(
        df_inst_sub[['hash_id', 'snapshot_year']], on='hash_id', how='inner')

# Load complete dataset
df_gt_all = pd.read_pickle(os.path.join(COMPLETE_DATASET_DIR, 'df_wiki_predictions_and_gt_cie.pkl'))
df_inst_all = pd.read_pickle(os.path.join(COMPLETE_DATASET_DIR, 'df_instances_v13.pkl'))
if 'snapshot_year' not in df_gt_all.columns:
    df_gt_all = df_gt_all.merge(
        df_inst_all[['hash_id', 'snapshot_year']], on='hash_id', how='inner')

print(f'Subsampled: {df_gt_sub.shape[0]:,} rows, {df_gt_sub.hash_id.nunique():,} instances')
print(f'Complete:   {df_gt_all.shape[0]:,} rows, {df_gt_all.hash_id.nunique():,} instances')

Subsampled: 36,245 rows, 3,500 instances
Complete:   3,887,196 rows, 233,113 instances


In [4]:
# Per-snapshot dataset stats
dataset_stats = []

for yr in SNAPSHOT_YEARS:
    # Complete dataset
    all_yr = df_gt_all[df_gt_all['snapshot_year'] == yr]
    all_instances = all_yr['hash_id'].nunique()
    all_operations = all_yr[['hash_id', 'tkgu_type', 'triple_head', 'triple_relation', 'triple_tail']].drop_duplicates().shape[0]

    # Subsampled test set
    sub_yr = df_gt_sub[df_gt_sub['snapshot_year'] == yr]
    sub_instances = sub_yr['hash_id'].nunique()
    sub_operations = sub_yr[['hash_id', 'tkgu_type', 'triple_head', 'triple_relation', 'triple_tail']].drop_duplicates().shape[0]

    dataset_stats.append({
        'year': yr,
        'all_instances': all_instances,
        'all_operations': all_operations,
        'sub_instances': sub_instances,
        'sub_operations': sub_operations,
    })

df_dataset = pd.DataFrame(dataset_stats)
df_dataset

,year,all_instances,all_operations,sub_instances,sub_operations
0,2019,37820,248944,500,4094
1,2020,31185,274540,500,5273
2,2021,40533,420229,500,7940
3,2022,30910,244929,500,4417
4,2023,26496,201224,500,4055
5,2024,32677,260930,500,5229
6,2025,33492,292802,500,5237


## Compute KG statistics

In [5]:
kg_stats = []

for yr in SNAPSHOT_YEARS:
    # Entity count from Wikipedia dictionary
    dict_path = os.path.join(DICT_DIR, f'{yr}-01-01-wiki_dictionary.jsonl')
    with open(dict_path) as f:
        n_entities = sum(1 for _ in f)

    # Triples + relation types from KG snapshot TSV
    tsv_path = os.path.join(KG_SNAPSHOTS_DIR, f'kg_snapshot_{yr}-01-01.tsv')
    rels = set()
    n_triples = 0
    with open(tsv_path) as f:
        for line in f:
            parts = line.split('\t')
            if len(parts) >= 3:
                rels.add(parts[1])
                n_triples += 1

    kg_stats.append({
        'year': yr,
        'entities': n_entities,
        'rel_types': len(rels),
        'triples': n_triples,
    })
    print(f'{yr}: entities={n_entities/1e6:.2f}M  rels={len(rels):,}  triples={n_triples/1e6:.2f}M')

df_kg = pd.DataFrame(kg_stats)
df_kg

2019: entities=5.96M  rels=962  triples=25.73M


2020: entities=6.14M  rels=1,068  triples=28.76M


2021: entities=6.34M  rels=1,165  triples=30.84M


2022: entities=6.54M  rels=1,219  triples=33.41M


2023: entities=6.67M  rels=1,275  triples=34.99M


2024: entities=6.80M  rels=1,297  triples=36.31M


2025: entities=6.93M  rels=1,351  triples=37.54M


,year,entities,rel_types,triples
0,2019,5961604,962,25729926
1,2020,6144832,1068,28762257
2,2021,6343586,1165,30844859
3,2022,6537564,1219,33413356
4,2023,6670403,1275,34985007
5,2024,6799879,1297,36306231
6,2025,6931228,1351,37542677


## Generate LaTeX table

In [6]:
def fmt_k(n):
    """Format as XK or X.XXK."""
    if n >= 1000:
        return f'{n/1000:.0f}K'
    return str(n)

def fmt_m(n):
    """Format as X.XXM."""
    return f'{n/1e6:.2f}M'

rows = []
for _, ds in df_dataset.iterrows():
    yr = ds['year']
    kg = df_kg[df_kg['year'] == yr].iloc[0]

    row = (
        f"{yr} & "
        f"{fmt_k(ds['all_instances'])} & {fmt_k(ds['all_operations'])} & "
        f"{ds['sub_instances']} & {fmt_k(ds['sub_operations'])} & "
        f"{fmt_m(kg['entities'])} & {kg['rel_types']:,} & {fmt_m(kg['triples'])} \\\\"
    )
    rows.append(row)

body = ' \n'.join(rows)

latex = f"""\\begin{{table}}[t]
\\caption{{Statistics of \\datasetname, organized by KG snapshots. For each snapshot, we report the number of \\textit{{instances}} and TKGU \\textit{{operations}} in both the \\textit{{complete dataset}} and the \\textit{{subsampled test set}}. The \\textit{{KG statistics}} section summarizes the number of entities, relation types, and triples in each KG snapshot.}}
\\label{{tab:dataset_statistics}}
  \\begin{{center}}
    \\begin{{small}}
      \\begin{{sc}}
\\begin{{tabular}}{{c cccc ccc}}
\\toprule
& \\multicolumn{{2}}{{c}}{{Complete dataset}} & \\multicolumn{{2}}{{c}}{{Subsampled test set}} & \\multicolumn{{3}}{{c}}{{KG statistics}} \\\\ 
\\cmidrule(lr){{2-3}}\\cmidrule(lr){{4-5}}\\cmidrule(l){{6-8}} 
Snapshot & Instances & Operations & Instances & Operations & Entities & Rel. Types & Triples  \\\\
\\midrule 
{body}
\\bottomrule
\\end{{tabular}}
      \\end{{sc}}
    \\end{{small}}
  \\end{{center}}
  \\vskip -0.1in
\\end{{table}}"""

print(latex)

\begin{table}[t]
\caption{Statistics of \datasetname, organized by KG snapshots. For each snapshot, we report the number of \textit{instances} and TKGU \textit{operations} in both the \textit{complete dataset} and the \textit{subsampled test set}. The \textit{KG statistics} section summarizes the number of entities, relation types, and triples in each KG snapshot.}
\label{tab:dataset_statistics}
  \begin{center}
    \begin{small}
      \begin{sc}
\begin{tabular}{c cccc ccc}
\toprule
& \multicolumn{2}{c}{Complete dataset} & \multicolumn{2}{c}{Subsampled test set} & \multicolumn{3}{c}{KG statistics} \\ 
\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(l){6-8} 
Snapshot & Instances & Operations & Instances & Operations & Entities & Rel. Types & Triples  \\
\midrule 
2019 & 38K & 249K & 500 & 4K & 5.96M & 962 & 25.73M \\ 
2020 & 31K & 275K & 500 & 5K & 6.14M & 1,068 & 28.76M \\ 
2021 & 41K & 420K & 500 & 8K & 6.34M & 1,165 & 30.84M \\ 
2022 & 31K & 245K & 500 & 4K & 6.54M & 1,219 & 33.41M \\ 

## Optional: Unfiltered KG relation types (full Wikidata)

The KG snapshot TSVs above are **filtered** to entities with Wikipedia pages
(`heads_in_wikipedia=True`, `tails_in_wikipedia=True` in the step 03 config).
This gives fewer relation types than the full Wikidata KG.

To compute unfiltered relation types per snapshot, you need to:
1. Run `s03_get_kg_snapshot` with `heads_in_wikipedia=false`, `tails_in_wikipedia=false`
   to produce unfiltered KG snapshot TSVs
2. Or load the raw Wikidata history graph (the `.pt` cache from step 03)

**Requirements:** ~128GB RAM salloc:
```
salloc -p rome -n 1 --ntasks-per-node 1 --cpus-per-task 1 -t 5:00:00 --mem=128G
```

If unfiltered KG snapshot TSVs exist, set `KG_SNAPSHOTS_UNFILTERED_DIR` below and run the cell.

In [7]:
# --- CONFIGURE THIS (optional) ---
# Set to the directory containing UNFILTERED kg_snapshot_YYYY-01-01.tsv files.
# These are produced by running s03_get_kg_snapshot with heads_in_wikipedia=false.
# Set to None to skip.
KG_SNAPSHOTS_UNFILTERED_DIR = None  # e.g. '/path/to/storage/.../unfiltered/'

if KG_SNAPSHOTS_UNFILTERED_DIR is not None:
    kg_unfiltered_stats = []
    for yr in SNAPSHOT_YEARS:
        tsv_path = os.path.join(KG_SNAPSHOTS_UNFILTERED_DIR, f'kg_snapshot_{yr}-01-01.tsv')
        rels = set()
        n_triples = 0
        with open(tsv_path) as f:
            for line in f:
                parts = line.split('\t')
                if len(parts) >= 3:
                    rels.add(parts[1])
                    n_triples += 1
        kg_unfiltered_stats.append({
            'year': yr,
            'rel_types_unfiltered': len(rels),
            'triples_unfiltered': n_triples,
        })
        print(f'{yr}: unfiltered rels={len(rels):,}  triples={n_triples/1e6:.2f}M')

    df_kg_unfiltered = pd.DataFrame(kg_unfiltered_stats)
    df_kg_unfiltered
else:
    print('KG_SNAPSHOTS_UNFILTERED_DIR not set — skipping unfiltered KG stats.')

KG_SNAPSHOTS_UNFILTERED_DIR not set — skipping unfiltered KG stats.
